# Hidden Sardinian Gems: scoprire le gemme nascoste della Sardegna

Questo progetto combina quattro indicatori di pressione turistica per trovare luoghi belli ma poco visitati, adattando la raccomandazione al mese dell'anno. Il notebook è **self-contained e riproducibile**: legge tutti i CSV disponibili, ricostruisce la geometria dei 377 comuni e ricalcola punteggi e visualizzazioni ogni volta che viene eseguito.

Il focus speciale è marino: il database curato include grotte (marine, terrestri e subacquee) e spiagge pure, con un bonus `marine_gem_bonus` che rende visibili anche le gemme costiere meno note. I risultati sono esplorativi e non sostituiscono informazioni ufficiali su accessibilità, sicurezza o tutela ambientale.

In [ ]:
# Installazione minima: il notebook resta eseguibile anche in un ambiente nuovo.
import subprocess, sys
for pkg in ['pandas', 'numpy', 'matplotlib']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import json, math, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

BASE_DIR = Path("../../")
ROOT = BASE_DIR / "data" / "mappa_finale"
TMP = ROOT / 'tmp'
OUT = ROOT / 'data' / 'sardegna-gems'
years = [2022, 2023, 2024, 2025]
months = list(range(1, 13))
month_names = ['Gennaio', 'Febbraio', 'Marzo', 'Aprile', 'Maggio', 'Giugno', 'Luglio', 'Agosto', 'Settembre', 'Ottobre', 'Novembre', 'Dicembre']
print('Ambiente pronto:', ROOT)

Ambiente pronto: ../../data/mappa_finale


## 1. Introduzione

L'idea di **Hidden Sardinian Gems** è separare la pressione turistica dalla qualità potenziale del territorio: un comune con overtourism basso, paesaggio costiero o montano, popolazione ridotta e infrastrutture non sovradimensionate può essere una buona candidata per un itinerario alternativo.

La stagionalità è esplicita: la stessa località può essere interessante in inverno e meno adatta in agosto. Il flusso completo è documentato qui, senza dipendere da variabili create in precedenza. Alla fine vengono anche rigenerati `data.json` e il JSON incorporato nell'HTML della mappa.

## 2. I 4 indicatori

Gli indicatori sono calcolati a partire da presenze, letti, superficie e popolazione:

1. **Densità Turistica** = `presenze / superficie_kmq` (presenze per km²).
2. **Densità Ricettiva** = `letti_totali / superficie_kmq` (letti per km²).
3. **Intensità Turistica** = `presenze / popolazione_residente` (turisti/residenti).
4. **Utilizzazione Lorda** = `(presenze_annue / (letti × 365)) × 100` (tasso di occupazione).

I CSV sono esportazioni mensili e possono contenere valori mancanti quando la copertura statistica non è sufficiente.

In [3]:
# Tutti i 20 CSV: utf-8-sig rimuove il BOM presente negli export.
overtourism = {y: pd.read_csv(TMP / f'indice_overtourism_{y}.csv', encoding='utf-8-sig') for y in years}
intensita = {y: pd.read_csv(TMP / f'intensita_turistica_{y}.csv', encoding='utf-8-sig') for y in years}
ricettiva = {y: pd.read_csv(TMP / f'densita_ricettiva_{y}.csv', encoding='utf-8-sig') for y in years}
densita = {y: pd.read_csv(TMP / f'densita_turistica_{y}.csv', encoding='utf-8-sig') for y in years}
util_lorda = {y: pd.read_csv(TMP / f'utilizzazione_lorda_{y}.csv', encoding='utf-8-sig') for y in years}
attrattivita = {pd.read_csv(TMP / f'indicatore_di_attrattivita.csv', encoding='utf-8-sig')}
print(f'Anni: {years}')
print(f'Comuni 2025: {overtourism[2025]["comune"].nunique()}')
print(f'Righe overtourism 2025: {len(overtourism[2025])}')

FileNotFoundError: [Errno 2] No such file or directory: '../../data/mappa_finale/tmp/indice_overtourism_2022.csv'

'/home/davide/Scrivania'

In [ ]:
# Una panoramica leggibile: prime righe e statistiche descrittive per indicatore.
for nome, frame in [('overtourism', overtourism[2025]), ('intensità', intensita[2025]), ('ricettiva', ricettiva[2025]), ('densità', densita[2025]), ('utilizzazione lorda', util_lorda[2025])]:
    print(f'\n--- {nome} ---')
    display(frame.head(3))
    display(frame.select_dtypes(include='number').describe().round(3))

## 3. Normalizzazione e indice di Overtourism

Per rendere confrontabili le scale, ogni indicatore viene limitato al percentile 99 (winsorization) e trasformato con min-max usando min = 0. In questa versione riproduciamo gli anchor fissati nei notebook di analisi:

| indicatore | min | cap p99 |
|---|---:|---:|
| densità turistica | 0 | 1710.44 |
| intensità turistica | 0 | 30.37 |
| densità ricettiva | 0 | 204.82 |
| utilizzazione lorda | 0 | 29.62 |

L'indice è la media geometrica dei valori normalizzati disponibili: `I = (n₁ × ... × nₖ)^(1/k)`. Se un valore disponibile è zero, l'indice è zero; le righe prive di tutti gli indicatori restano mancanti.

In [ ]:
ANCHORS = {
    'densita_turistica': 1710.44,
    'intensita_turistica': 30.37,
    'densita_ricettiva': 204.82,
    'utilizzazione_lorda': 29.62,
}

def norm_fixed(series, cap):
    # Winsorization al cap p99 e min-max con minimo fisso 0.
    return series.clip(lower=0, upper=cap) / cap

# Verifica degli anchor contro i campi *_norm esportati nel CSV dell'indice.
raw_2025 = overtourism[2025].copy()
checks = {}
for raw_col, norm_col, key in [
    ('densita_turistica', 'densita_turistica_norm', 'densita_turistica'),
    ('intensita_turistica_presenze', 'intensita_turistica_norm', 'intensita_turistica'),
    ('densita_ricettiva', 'densita_ricettiva_norm', 'densita_ricettiva'),
    ('utilizzazione_lorda_pct', 'utilizzazione_lorda_norm', 'utilizzazione_lorda'),
]:
    calc = norm_fixed(pd.to_numeric(raw_2025[raw_col], errors='coerce'), ANCHORS[key])
    exported = pd.to_numeric(raw_2025[norm_col], errors='coerce')
    checks[key] = {'max_abs_error': float((calc-exported).abs().max()), 'rows_compared': int(exported.notna().sum())}
print(pd.DataFrame(checks).T)
assert all(v['max_abs_error'] < 1e-3 for v in checks.values())

In [ ]:
# Verifica indipendente della media geometrica dell'indice.
norm_cols = ['densita_turistica_norm', 'intensita_turistica_norm', 'densita_ricettiva_norm', 'utilizzazione_lorda_norm']
def geometric_mean_row(row):
    vals = pd.to_numeric(row[norm_cols], errors='coerce').dropna().to_numpy(float)
    if len(vals) == 0:
        return np.nan
    if np.any(vals == 0):
        return 0.0
    return float(np.prod(vals) ** (1 / len(vals)))
recomputed = raw_2025.apply(geometric_mean_row, axis=1)
exported_index = pd.to_numeric(raw_2025['indice_overtourism'], errors='coerce')
max_error = float((recomputed-exported_index).abs().max())
print(f'Errore massimo media geometrica: {max_error:.6f}')
assert max_error < 1e-3

plt.figure(figsize=(8, 4))
plt.hist(exported_index.dropna(), bins=30, color='#287c8e', edgecolor='white')
plt.title('Distribuzione dell’indice di overtourism — 2025')
plt.xlabel('Indice di overtourism'); plt.ylabel('Numero di righe'); plt.tight_layout(); plt.show()

## 4. Caricamento della geometria

Usiamo il GeoJSON locale derivato dal layer openpolis/geojson-italy per i confini dei comuni sardi e il GeoJSON regionale per il contorno. I centroidi sono calcolati con una media equirettangolare semplice; l'area è stimata proiettando i gradi in km. Questa scelta è coerente con il builder esistente ed è adeguata allo scopo di visualizzazione e ranking, non a misure catastali.

In [ ]:
with open(OUT / 'sardinia_municipalities.geojson', encoding='utf-8') as f:
    geo = json.load(f)
with open(OUT / 'sardinia_region.geojson', encoding='utf-8') as f:
    region_geo = json.load(f)

def iter_rings(geometry):
    # Gestisce Polygon e MultiPolygon GeoJSON.
    if geometry['type'] == 'Polygon':
        yield from geometry['coordinates']
    else:
        for polygon in geometry['coordinates']:
            yield from polygon

def centroid_area(geometry):
    points = [p for ring in iter_rings(geometry) for p in ring[:-1]]
    if not points:
        return 40.1, 9.1, 1.0
    lng = sum(p[0] for p in points) / len(points)
    lat = sum(p[1] for p in points) / len(points)
    sx, sy = 111.32 * math.cos(math.radians(lat)), 111.32
    area = 0.0
    for ring in iter_rings(geometry):
        xy = [(p[0]*sx, p[1]*sy) for p in ring]
        area += abs(sum(xy[i][0]*xy[(i+1)%len(xy)][1] - xy[(i+1)%len(xy)][0]*xy[i][1] for i in range(len(xy)-1))) / 2
    return lat, lng, max(0.2, area)

comuni_geo = {}
for feat in geo['features']:
    name = feat['properties'].get('name') or feat['properties'].get('name_it')
    lat, lng, area = centroid_area(feat['geometry'])
    comuni_geo[name] = {'name': name, 'lat': lat, 'lng': lng, 'area_kmq': area, 'geometry': feat['geometry']}
print('Feature GeoJSON:', len(comuni_geo))
assert len(comuni_geo) == 377

In [ ]:
# La costa è stimata come prossimità al perimetro regionale + alcuni comuni
# costieri esplicitamente presenti nel database marino. Non si usa una lista
# segreta: la soglia è visibile e modificabile.
def region_rings():
    for feat in region_geo['features']:
        yield from iter_rings(feat['geometry'])
region_boundary = [p for ring in region_rings() for p in ring]
def point_segment_distance(lat, lng, a, b):
    # Distanza approssimata in gradi con longitudine corretta alla latitudine.
    x, y = lng * math.cos(math.radians(lat)), lat
    ax, ay = a[0] * math.cos(math.radians(lat)), a[1]
    bx, by = b[0] * math.cos(math.radians(lat)), b[1]
    dx, dy = bx-ax, by-ay
    t = 0 if dx*dx+dy*dy == 0 else max(0, min(1, ((x-ax)*dx+(y-ay)*dy)/(dx*dx+dy*dy)))
    return math.hypot(x-(ax+t*dx), y-(ay+t*dy))
def near_region_boundary(lat, lng, threshold=0.18):
    return min(point_segment_distance(lat, lng, region_boundary[i], region_boundary[i+1]) for i in range(len(region_boundary)-1)) < threshold

# Hint iniziale: sono tutti comuni che compaiono nella banca marina e quindi
# devono essere costieri per la logica del progetto.
COASTAL_NAME_HINTS = {'Alghero','Dorgali','Baunei','La Maddalena','Gairo','Cabras','Teulada','Domus de Maria',"Sant'Anna Arresi",'Tresnuraghes','Santa Teresa Gallura',"Trinità d'Agultu e Vignola"}
for meta in comuni_geo.values():
    meta['coastal'] = bool(near_region_boundary(meta['lat'], meta['lng']) or meta['name'] in COASTAL_NAME_HINTS)
    meta['mountain'] = meta['name'] in {'Gairo','Ulassai','Fonni','Desulo','Orgosolo','Urzulei','Baunei'}
coastal_table = pd.DataFrame([{'tipo': 'costiero' if m['coastal'] else 'interno', 'comuni': 1} for m in comuni_geo.values()]).groupby('tipo').sum()
display(coastal_table)

## 5. Il Database delle Grotte e Spiagge Pure

La Sardegna custodisce alcune delle grotte marine e terrestri più spettacolari del Mediterraneo, oltre a spiagge incontaminate accessibili solo via mare o con lunghe escursioni. Questa sezione costruisce un database curato di queste località e le collega ai comuni corrispondenti.

Le due liste seguenti sono volutamente Python puro e facilmente editabili: per aggiungere una località basta copiare un dizionario e mantenere `name`, `comune`, coordinate, descrizione e `bonus`. I bonus sono una scelta editoriale trasparente, non una misura ufficiale.

In [ ]:
# Database editabile delle grotte: coordinate in gradi decimali.
GROTTE = [
    {'name':'Grotta di Nettuno','comune':'Alghero','lat':40.566,'lng':8.162,'type':'marina','description':'Una delle più grandi cavità marine d’Italia, stalattiti e lago sotterraneo','bonus':0.35},
    {'name':'Grotta del Bue Marino','comune':'Dorgali','lat':40.279,'lng':9.600,'type':'marina','description':'Ex rifugio della foca monaca, graffiti neolitici, accessibile via mare da Cala Gonone','bonus':0.30},
    {'name':'Grotta di Ispinigoli','comune':'Dorgali','lat':40.260,'lng':9.530,'type':'terrestre','description':'Stalagmite di 38m tra le più alte d’Europa, abisso delle Vergini','bonus':0.20},
    {'name':'Grotta Su Marmuri','comune':'Ulassai','lat':39.510,'lng':9.590,'type':'terrestre','description':'850m di gallerie pianeggianti, colonne di calcite come sculture di marmo','bonus':0.20},
    {'name':'Grotta di Su Mannau','comune':'Fluminimaggiore','lat':39.440,'lng':8.400,'type':'terrestre','description':'Complesso carsico con fiumi sotterranei e reperti nuragici','bonus':0.20},
    {'name':'Grotta del Fico','comune':'Baunei','lat':40.190,'lng':9.660,'type':'marina','description':'Rifugio della foca monaca, fico sospeso sulla parete a picco sul mare','bonus':0.30},
    {'name':'Grotta di Nereo','comune':'Alghero','lat':40.560,'lng':8.150,'type':'subacquea','description':'La grotta sommersa più grande del Mediterraneo, 300m di galleria principale','bonus':0.25},
    {'name':'Grotta Verde','comune':'Alghero','lat':40.550,'lng':8.170,'type':'marina','description':'Graffiti paleolitici, laghetto dai riflessi verdi, rocce di 200 milioni di anni fa','bonus':0.25},
    {'name':'Grotta Su Meraculu (Miracolo)','comune':'Baunei','lat':40.170,'lng':9.670,'type':'marina','description':'Dietro Cala Sisine, sculture naturali e giochi di luce','bonus':0.25},
    {'name':'Grotte Is Zuddas','comune':'Santadi','lat':39.070,'lng':8.620,'type':'terrestre','description':'Concrezioni di aragonite millenarie, eccentriche rare','bonus':0.15},
]
# Database editabile delle spiagge pure.
SPIAGGE_PURE = [
    {'name':'Cala Luna','comune':'Baunei','lat':40.190,'lng':9.640,'description':'Monumento nazionale, grottoni sulla spiaggia, laghetto di oleandri','bonus':0.35},
    {'name':'Cala Mariolu','comune':'Baunei','lat':40.220,'lng':9.650,'description':'Spiaggia di sassi bianchi levigati, acqua turchese del Golfo di Orosei','bonus':0.30},
    {'name':'Cala Goloritzé','comune':'Baunei','lat':40.150,'lng':9.670,'description':'Patrimonio UNESCO, arco naturale e Aguglia di 143m','bonus':0.35},
    {'name':'Cala Sisine','comune':'Baunei','lat':40.170,'lng':9.680,'description':'Foce di una gola montana, ciottoli bianchi, raggiungibile via mare o trekking','bonus':0.30},
    {'name':'Cala Coticcio','comune':'La Maddalena','lat':41.060,'lng':9.410,'description':'La Tahiti della Sardegna, baia paradisiaca su Caprera, accesso limitato','bonus':0.35},
    {'name':'Cala Brigantina','comune':'La Maddalena','lat':41.070,'lng':9.420,'description':'Due calette di sabbia bianca su Caprera, bassi fondali, snorkeling','bonus':0.30},
    {'name':'Spiaggia Su Sirboni','comune':'Gairo','lat':39.760,'lng':9.480,'description':'Marina di Gairo, sabbia bianca e scogli rossi, villaggio abbandonato alle spalle','bonus':0.30},
    {'name':'Oasi di Seu','comune':'Cabras','lat':39.950,'lng':8.430,'description':'Area marina protetta del Sinis, relitto sommerso, acqua cristallina','bonus':0.25},
    {'name':'Tuerredda','comune':'Teulada','lat':38.900,'lng':8.850,'description':'La piccola Tahiti della Sardegna, sfumature caraibiche','bonus':0.25},
    {'name':'Cala Cipolla','comune':'Domus de Maria','lat':38.910,'lng':8.880,'description':'Tra dune e scogliere, vento maentiche, acqua trasparente','bonus':0.25},
    {'name':'Porto Pino','comune':"Sant'Anna Arresi",'lat':39.140,'lng':8.770,'description':'Dune di sabbia bianca altissime, poco sfruttata dal turismo di massa','bonus':0.30},
    {'name':'Pedra Mar','comune':'Tresnuraghes','lat':40.350,'lng':8.530,'description':'Costa della Planargia, solitudine lontana dal caos','bonus':0.25},
    {'name':'Cala Balcaccia','comune':'Santa Teresa Gallura','lat':41.220,'lng':9.180,'description':'Sabbia bianca e scogli di granito, isolata anche in alta stagione','bonus':0.25},
    {'name':'Costa Paradiso','comune':"Trinità d'Agultu e Vignola",'lat':41.050,'lng':9.170,'description':'Sabbia rossa, scogliere modellate dal vento, mare turchese, snorkeling','bonus':0.25},
]
MARINE_GEMS = GROTTE + SPIAGGE_PURE
print(f'Grotte: {len(GROTTE)} | Spiagge pure: {len(SPIAGGE_PURE)} | Totale: {len(MARINE_GEMS)}')
assert len(GROTTE) >= 10 and len(SPIAGGE_PURE) >= 14

In [ ]:
# Collegamento ai comuni e massimo bonus per comune.
marine_by_comune = {}
for gem in MARINE_GEMS:
    if gem['comune'] not in comuni_geo:
        print('Attenzione: comune non trovato nella geometria:', gem['comune'])
    marine_by_comune.setdefault(gem['comune'], []).append(gem)
marine_bonus = {name: max(g['bonus'] for g in gems) for name, gems in marine_by_comune.items()}
marine_table = pd.DataFrame([
    {'comune': name, 'marine_gem_bonus': marine_bonus[name], 'cosa_contiene': '; '.join(g['name'] for g in gems)}
    for name, gems in sorted(marine_by_comune.items())
])
display(marine_table)
print('Comuni marini riconosciuti:', len(marine_table))

## 6. L'algoritmo “Hidden Gem”

Il punteggio completo è:

1. `BASE_SECLUSION = 1 - overtourism` (più nascosto = più gemma).
2. **Quality multiplier**: ×1.3 per comuni costieri; fattore popolazione (pop < 1.000 → 1.3, < 3.000 → 1.15, < 5.000 → 1.05, altrimenti 1); fattore letti (0 → 0.3, 1–50 → 1, 51–200 → 0.85, oltre 200 → 0.6).
3. **Seasonal adjustment**: inverno costa +0.20/montagna −0.20; primavera tutti +0.10 e interno +0.15; estate montagna interna +0.25 e hub costieri −0.15; autunno costa +0.15 e comuni agricoli +0.10.
4. **Marine gem bonus**: si aggiunge il massimo bonus di grotte/spiagge del comune.
5. Elegibilità: almeno 2 indicatori, presenze > 0 e overtourism < p75 mensile.

`HIDDEN_GEM_SCORE = BASE_SECLUSION × QUALITY × (1 + SEASONAL) × (1 + MARINE_GEM_BONUS)`. La visualizzazione normalizza il valore a 0–100 per leggibilità.

In [ ]:
# Preparo i campi comunali da 2025: i valori sono ripetuti mensilmente nei CSV.
def first_by_comune(frame):
    return frame.sort_values(['comune','mese']).groupby('comune', as_index=False).first()
int25 = first_by_comune(intensita[2025])
ric25 = first_by_comune(ricettiva[2025])
util25 = first_by_comune(util_lorda[2025])
base = pd.DataFrame({'comune': sorted(comuni_geo)})
for frame, cols in [(int25, ['popolazione_residente']), (ric25, ['letti_totali','superficie_kmq']), (util25, ['presenze_annue'])]:
    keep = ['comune'] + [c for c in cols if c in frame.columns]
    base = base.merge(frame[keep], on='comune', how='left')
base['popolazione_residente'] = pd.to_numeric(base['popolazione_residente'], errors='coerce').fillna(0)
base['letti_totali'] = pd.to_numeric(base['letti_totali'], errors='coerce').fillna(0)
base['presenze_annue'] = pd.to_numeric(base['presenze_annue'], errors='coerce').fillna(0)
base['coastal'] = base['comune'].map(lambda n: comuni_geo[n]['coastal'])
base['mountain'] = base['comune'].map(lambda n: comuni_geo[n]['mountain'])
base['marine_gem_bonus'] = base['comune'].map(marine_bonus).fillna(0.0)
base['marine_gems'] = base['comune'].map(lambda n: '; '.join(g['name'] for g in marine_by_comune.get(n, [])))
base.head()

In [ ]:
# Regole stagionali e fattori di qualità: ogni passaggio è esplicito.
AGRICULTURAL = {'Arborea','Barumini','Cabras','Mogoro','Morgongiori','Oristano','Pau','Siddi','Siamanna','Sini','Terralba','Uras','Villaurbana'}
def size_factor(pop):
    if pop < 1000: return 1.3
    if pop < 3000: return 1.15
    if pop < 5000: return 1.05
    return 1.0
def bed_factor(beds):
    if beds == 0: return 0.3
    if beds <= 50: return 1.0
    if beds <= 200: return 0.85
    return 0.6
def seasonal_adjustment(month, coastal, mountain, name, beds):
    if month in (12, 1, 2):
        return 0.20 if coastal else (-0.20 if mountain else 0.0)
    if month in (3, 4, 5):
        return 0.10 + (0.15 if not coastal else 0.0)
    if month in (6, 7, 8):
        return 0.25 if (mountain and not coastal) else (-0.15 if coastal and beds > 200 else 0.0)
    return (0.15 if coastal else 0.0) + (0.10 if name in AGRICULTURAL else 0.0)

def score_one(row, month, overt):
    if pd.isna(overt) or row['popolazione_residente'] < 0:
        return 0.0, False, 0.0
    n_ind = row['n_indicatori_disponibili']
    eligible = bool(n_ind >= 2 and row['presenze'] > 0 and overt < row['p75_overtourism'])
    seasonal = seasonal_adjustment(month, row['coastal'], row['mountain'], row['comune'], row['letti_totali'])
    quality = (1.3 if row['coastal'] else 1.0) * size_factor(row['popolazione_residente']) * bed_factor(row['letti_totali'])
    score = (1-overt) * quality * (1+seasonal) * (1+row['marine_gem_bonus']) if eligible else 0.0
    return float(score), eligible, float(seasonal)

# Calcolo tutti i 377 comuni × 48 mese-anno, con p75 per ogni mese e anno.
records = []
for year in years:
    idx = overtourism[year].copy()
    idx['comune'] = idx['comune'].astype(str)
    for month in months:
        part = idx[idx['mese'].eq(month)].copy()
        p75 = pd.to_numeric(part['indice_overtourism'], errors='coerce').quantile(0.75)
        dmonth = densita[year][densita[year]['mese'].eq(month)][['comune','presenze_mese']].copy()
        dmonth['presenze'] = pd.to_numeric(dmonth['presenze_mese'], errors='coerce').fillna(0)
        joined = base.merge(part, on='comune', how='left', suffixes=('','_idx')).merge(dmonth[['comune','presenze']], on='comune', how='left')
        joined['presenze'] = joined['presenze'].fillna(0)
        joined['p75_overtourism'] = p75
        for _, r in joined.iterrows():
            overt = pd.to_numeric(r.get('indice_overtourism'), errors='coerce')
            sc, eligible, seasonal = score_one(r, month, overt)
            records.append({'year':year,'month':month,'comune':r['comune'],'score':sc*100,'raw_score':sc,'overtourism':float(overt) if pd.notna(overt) else np.nan,'eligible':eligible,'coastal':bool(r['coastal']),'marine_gem_bonus':float(r['marine_gem_bonus']),'seasonal_adjustment':seasonal,'population':int(r['popolazione_residente']),'beds':float(r['letti_totali']),'presenze':float(r['presenze']),'n_indicatori':float(r.get('n_indicatori_disponibili',0) or 0),'marine_gems':r['marine_gems']})
scores = pd.DataFrame(records)
# Rank solo tra le righe eleggibili, per mese e anno.
scores['rank'] = scores.groupby(['year','month'])['score'].rank(method='min', ascending=False).where(scores['eligible'])
print('Righe calcolate:', len(scores), '| attese:', 377*48)
assert len(scores) == 377*48

## 7. Risultati — Le Gemme Nascoste

Il default è agosto 2025, ma tutte le tabelle sono derivate da `scores` e quindi cambiano se si sostituiscono i CSV o si modifica il database marino.

In [ ]:
def result_table(year, month, n=20, marine_only=False):
    q = scores[(scores.year==year)&(scores.month==month)&(scores.eligible)].copy()
    if marine_only: q = q[q.marine_gem_bonus > 0]
    q = q.sort_values(['score','comune'], ascending=[False,True]).head(n).copy()
    q.insert(0, 'Rank', range(1, len(q)+1))
    q['Score (0-100)'] = q['score'].round(2)
    q['overtourism'] = q['overtourism'].round(4)
    q['Costiero'] = np.where(q['coastal'], 'sì', 'no')
    q['Popolazione'] = q['population']
    q['Letti'] = q['beds'].round(0).astype(int)
    q['Gemme marine'] = q['marine_gems'].replace('', '—')
    return q[['Rank','comune','Score (0-100)','overtourism','Costiero','Popolazione','Letti','marine_gem_bonus','Gemme marine']]
display(result_table(2025, 8, 20))

In [ ]:
# Confronto stagionale: gennaio e agosto 2025.
winter = result_table(2025, 1, 10)[['Rank','comune','Score (0-100)','Costiero','Gemme marine']]
summer = result_table(2025, 8, 10)[['Rank','comune','Score (0-100)','Costiero','Gemme marine']]
print('TOP 10 — GENNAIO 2025'); display(winter)
print('TOP 10 — AGOSTO 2025'); display(summer)

In [ ]:
# Solo comuni con grotte/spiagge: ranking per inverno, primavera, estate e autunno.
marine_rows = []
for month, label in [(1,'inverno'),(4,'primavera'),(8,'estate'),(10,'autunno')]:
    t = result_table(2025, month, 10, marine_only=True)
    t.insert(1, 'stagione', label)
    marine_rows.append(t)
display(pd.concat(marine_rows, ignore_index=True))

## 8. Visualizzazioni

Le figure usano una palette mediterranea: blu mare, turchese, corallo e sabbia. I comuni con almeno una grotta o spiaggia pura sono evidenziati in corallo.

In [ ]:
MEDITERRANEO = {'marine':'#d95f59', 'coastal':'#287c8e', 'inland':'#e6a65d', 'grid':'#d8e5e7'}
def top_plot_data(year, month, n=15):
    return scores[(scores.year==year)&(scores.month==month)&(scores.eligible)].sort_values('score', ascending=False).head(n).sort_values('score')
fig, axes = plt.subplots(1, 3, figsize=(19, 6), gridspec_kw={'width_ratios':[1,1,1.15]})
for ax, month, title in [(axes[0],8,'Top 15 — Agosto 2025'), (axes[1],1,'Top 15 — Gennaio 2025')]:
    q = top_plot_data(2025, month)
    colors = [MEDITERRANEO['marine'] if x>0 else (MEDITERRANEO['coastal'] if c else MEDITERRANEO['inland']) for x,c in zip(q.marine_gem_bonus,q.coastal)]
    ax.barh(q.comune, q.score, color=colors)
    ax.set_title(title); ax.set_xlabel('Punteggio 0–100'); ax.grid(axis='x', alpha=.3)
q = scores[(scores.year==2025)&(scores.month==8)&(scores.overtourism.notna())]
axes[2].scatter(q.overtourism, q.score, c=np.where(q.marine_gem_bonus>0,MEDITERRANEO['marine'],MEDITERRANEO['coastal']), alpha=.6, s=20)
axes[2].set_title('Overtourism vs gem score — agosto 2025'); axes[2].set_xlabel('Overtourism'); axes[2].set_ylabel('Score 0–100'); axes[2].grid(alpha=.3)
fig.suptitle('Hidden Sardinian Gems — palette mediterranea', fontsize=15)
plt.tight_layout(); plt.show()

## 9. Rigenerazione della mappa HTML

L'ultima parte aggiorna il dataset usato dalla mappa esistente. Mantiene la struttura originale (`meta`, `region`, `comuni`, `monthly`) e aggiunge per ogni comune `marine_gem_bonus` e `marine_gems`. Per rendere l'HTML autosufficiente, lo stesso JSON viene inserito tra i tag `<script id="embedded-data">`.

In [ ]:
# Patch del payload della mappa con i risultati appena ricalcolati.
data_path = OUT / 'data.json'
index_path = OUT / 'index.html'
with open(data_path, encoding='utf-8') as f:
    payload = json.load(f)

# Indici rapidi per i risultati; un record per comune e mese/anno.
score_lookup = {(r.year,r.month,r.comune): r for r in scores.itertuples()}
for comune in payload.get('comuni', []):
    name = comune['name']
    comune['marine_gem_bonus'] = float(marine_bonus.get(name, 0.0))
    comune['marine_gems'] = [dict(g) for g in marine_by_comune.get(name, [])]
    for monthly in comune.get('monthly', []):
        r = score_lookup.get((monthly['year'], monthly['month'], name))
        if r is not None:
            monthly['gem_score'] = round(float(r.raw_score), 6)
            monthly['gem_score_norm'] = round(float(r.score), 4)
            monthly['eligible'] = bool(r.eligible)
            monthly['seasonal_adjustment'] = round(float(r.seasonal_adjustment), 4)
            monthly['gem_rank'] = int(r.rank) if pd.notna(r.rank) else None
payload['meta']['algorithm'] = 'base_seclusion × quality × (1 + seasonal_adjustment) × (1 + marine_gem_bonus); eligibility: n_indicatori >= 2, presenze > 0, overtourism < p75 mensile'
payload['meta']['marine_focus'] = {'grotte': len(GROTTE), 'spiagge_pure': len(SPIAGGE_PURE), 'comuni_con_gemme': len(marine_bonus)}
with open(data_path, 'w', encoding='utf-8') as f:
    json.dump(payload, f, ensure_ascii=False, separators=(',', ':'))

html = index_path.read_text(encoding='utf-8')
open_tag = '<script type="application/json" id="embedded-data">'
close_tag = '</script>'
start = html.find(open_tag)
if start == -1:
    raise RuntimeError('Tag embedded-data non trovato in index.html')
content_start = start + len(open_tag)
content_end = html.find(close_tag, content_start)
if content_end == -1:
    raise RuntimeError('Chiusura embedded-data non trovata in index.html')
embedded = json.dumps(payload, ensure_ascii=False, separators=(',', ':')).replace('</', '<\/')
index_path.write_text(html[:content_start] + embedded + html[content_end:], encoding='utf-8')
print(f'Rigenerati: {data_path} ({data_path.stat().st_size:,} byte)')
print(f'Aggiornato: {index_path} ({index_path.stat().st_size:,} byte)')

In [ ]:
# Verifica finale del deliverable: JSON leggibile e HTML con dati marini incorporati.
with open(data_path, encoding='utf-8') as f:
    verified = json.load(f)
html_check = index_path.read_text(encoding='utf-8')
assert len(verified['comuni']) == 377
assert all('marine_gem_bonus' in c and 'marine_gems' in c for c in verified['comuni'])
assert 'marine_gem_bonus' in html_check
assert html_check.count(open_tag) == 1 and html_check.count(close_tag) >= 1
print("PASS — data.json valido, 377 comuni patchati e marine_gem_bonus presente nell HTML.")

## 10. Conclusioni e prossimi passi

Il notebook dimostra un flusso riproducibile end-to-end: caricamento dei 20 CSV, verifica della normalizzazione e della media geometrica, geometria dei comuni, database marino, ranking stagionale, visualizzazioni e rigenerazione della mappa.

Per aggiornare il progetto basta sostituire i CSV in `/workspace/tmp/` mantenendo i nomi attesi e rieseguire il notebook. Possibili estensioni: aggiungere altre grotte e spiagge con fonti verificate, introdurre valutazioni degli utenti, integrare foto e tracce di accesso, distinguere meglio i vincoli di tutela e collegare dati meteo o di qualità dell'acqua. Le descrizioni editoriali qui presenti vanno considerate un punto di partenza da verificare prima di una pubblicazione turistica.